## importing libraries and dataset

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

In [6]:

weather_file = Path(
    "//Users/rohankhanna/Documents/track3_agritech_dataset_files/track3_weather_sensors.xlsx"
   
)

# Update the path above if you moved the file.
workbook = pd.ExcelFile(weather_file)
print("Worksheets:", workbook.sheet_names)

df_weather = pd.read_excel(workbook, sheet_name="sensor_logs")
df_weather_clean = df_weather.copy(deep=True)

print("Rows:", len(df_weather_clean))
print("Exact duplicates:", df_weather_clean.duplicated().sum())

display(df_weather_clean.head(10))

Worksheets: ['sensor_logs']
Rows: 15000
Exact duplicates: 0


,sensor_id,timestamp,temperature,temp_unit,rainfall,rain_unit,humidity_percent
0,SEN047,28/08/2026,68,f,43.80,mm,46.0
1,SEN034,2026-07-03 07:08:18 IST,85.3,°F,1.88,in,55.0
2,SEN020,NaN,18.9,Celsius,19.90,MM,NaN
3,SEN035,2026-07-09 11:08:41 UTC,71.8,Fahrenheit,0.91,inch,49.0
4,SEN048,2026-06-09 13:06:26 IST,39.8°C,NaN,43.90,mm,81.0
5,SEN035,17/03/2026,21.8°C,NaN,28.90,MM,47.0
6,SEN013,07-01-2026 05:48 PM,75,f,32.40,mm,43.0
7,SEN040,2026-06-12 09:52:51 IST,92.5,°F,NaN,NaN,92.0
8,SEN027,2026-06-02 16:00:04 UTC,22.1,°C,-48.10,mm,36.0
9,SEN034,NaN,30.3°C,NaN,NaN,NaN,51.0


## Inspecting missing values

In [7]:
inspection = df_weather_clean.replace(r"^\s*$", pd.NA, regex=True)

display(
    pd.DataFrame({
        "Data type": df_weather_clean.dtypes.astype(str),
        "Missing count": inspection.isna().sum()
    })
)

,Data type,Missing count
sensor_id,str,0
timestamp,str,1555
temperature,object,0
temp_unit,str,2263
rainfall,float64,791
rain_unit,str,791
humidity_percent,float64,1500


## Inspecting temperatures

In [8]:
print("Unit labels:")
display(
    df_weather_clean["temp_unit"]
    .value_counts(dropna=False)
    .to_frame("row_count")
)

missing_unit = (
    df_weather_clean["temp_unit"]
    .astype("string")
    .fillna("")
    .str.strip()
    .eq("")
)

print("\nTemperatures with embedded units:")
display(
    df_weather_clean.loc[
        missing_unit, ["temperature", "temp_unit"]
    ].head(15)
)

Unit labels:


,row_count
temp_unit,
NaN,2263
c,1733
°C,1683
C,1679
Celsius,1622
f,1551
°F,1539
F,1466
Fahrenheit,1464



Temperatures with embedded units:


,temperature,temp_unit
4,39.8°C,NaN
5,21.8°C,NaN
9,30.3°C,NaN
18,31.8°C,NaN
22,24.6°C,NaN
28,28.4°C,NaN
32,37.6°C,NaN
58,38.6°C,NaN
59,35.6°C,NaN
60,31.2°C,NaN


## Inspecting numbers and standardizing units

In [9]:
text = (
    df_weather_clean["temperature"]
    .astype("string").str.strip().str.casefold()
)

parts = text.str.extract(
    r"^([+-]?(?:\d+(?:\.\d*)?|\.\d+))\s*(°?\s*[cf]|celsius|fahrenheit)?$"
)

df_weather_clean["temperature_numeric"] = pd.to_numeric(
    parts[0], errors="coerce"
)

unit_mapping = {
    "c": "C", "°c": "C", "celsius": "C",
    "f": "F", "°f": "F", "fahrenheit": "F"
}

column_label = (
    df_weather_clean["temp_unit"]
    .astype("string").str.strip().str.casefold()
    .str.replace(r"\s+", "", regex=True)
    .replace("", pd.NA)
)

embedded_label = parts[1].str.replace(r"\s+", "", regex=True)

column_unit = column_label.map(unit_mapping)
embedded_unit = embedded_label.map(unit_mapping)

conflict = (
    column_unit.notna()
    & embedded_unit.notna()
    & column_unit.ne(embedded_unit)
)

unknown = (
    (column_label.notna() & column_unit.isna())
    | (embedded_label.notna() & embedded_unit.isna())
)

df_weather_clean["temperature_unit_clean"] = (
    column_unit.fillna(embedded_unit).mask(conflict | unknown)
)

print(
    "Unparseable temperatures:",
    df_weather_clean["temperature_numeric"].isna().sum()
)
print("Conflicting units:", conflict.sum())
print("Unrecognized unit labels:", unknown.sum())

display(
    df_weather_clean["temperature_unit_clean"]
    .value_counts(dropna=False)
    .to_frame("row_count")
)

Unparseable temperatures: 0
Conflicting units: 0
Unrecognized unit labels: 0


,row_count
temperature_unit_clean,
C,8980
F,6020


## Converting fahrenheit to celsius

In [10]:
df_weather_clean["temperature_c"] = (
    df_weather_clean["temperature_numeric"].copy()
)

fahrenheit = df_weather_clean["temperature_unit_clean"].eq("F")

df_weather_clean.loc[fahrenheit, "temperature_c"] = (
    df_weather_clean.loc[fahrenheit, "temperature_numeric"] - 32
) * 5 / 9

display(
    df_weather_clean[
        ["temperature", "temperature_unit_clean", "temperature_c"]
    ].head(10)
)

print("Celsius temperature summary:")
display(df_weather_clean["temperature_c"].describe())

,temperature,temperature_unit_clean,temperature_c
0,68,F,20.0
1,85.3,F,29.611111
2,18.9,C,18.9
3,71.8,F,22.111111
4,39.8°C,C,39.8
5,21.8°C,C,21.8
6,75,F,23.888889
7,92.5,F,33.611111
8,22.1,C,22.1
9,30.3°C,C,30.3


Celsius temperature summary:


count      15000.0
mean     27.430508
std       7.186954
min           15.0
25%      21.222222
50%           27.5
75%           33.6
max           40.0
Name: temperature_c, dtype: Float64

## Inspecting rainfall units and values

In [11]:
print("Rainfall units:")
display(
    df_weather_clean["rain_unit"]
    .value_counts(dropna=False)
    .to_frame("row_count")
)

print("\nSample rainfall values:")
display(
    df_weather_clean[["rainfall", "rain_unit"]].head(15)
)

print("\nRainfall summary before conversion:")
display(df_weather_clean["rainfall"].describe())

print(
    "Negative rainfall values:",
    df_weather_clean["rainfall"].lt(0).sum()
)

Rainfall units:


,row_count
rain_unit,
mm,4459
MM,3038
millimeters,2916
inches,1285
in,1270
inch,1241
NaN,791



Sample rainfall values:


,rainfall,rain_unit
0,43.80,mm
1,1.88,in
2,19.90,MM
3,0.91,inch
4,43.90,mm
5,28.90,MM
6,32.40,mm
7,NaN,NaN
8,-48.10,mm
9,NaN,NaN



Rainfall summary before conversion:


count    14209.000000
mean        13.176203
std         20.879858
min        -50.000000
25%          0.980000
50%         10.000000
75%         29.900000
max         50.000000
Name: rainfall, dtype: float64

Negative rainfall values: 1518


## Making negative and missing values Flagged

In [12]:
unit = (
    df_weather_clean["rain_unit"]
    .astype("string").str.strip().str.casefold()
)

df_weather_clean["rain_unit_clean"] = unit.map({
    "mm": "mm",
    "millimeters": "mm",
    "in": "inches",
    "inch": "inches",
    "inches": "inches"
})

rain = df_weather_clean["rainfall"]

df_weather_clean["rainfall_status"] = np.select(
    [
        rain.isna().to_numpy(dtype=bool),
        rain.lt(0).fillna(False).to_numpy(dtype=bool),
        df_weather_clean["rain_unit_clean"].isna().to_numpy(dtype=bool)
    ],
    [
        "Missing rainfall",
        "Negative rainfall",
        "Missing or unrecognized unit"
    ],
    default="Valid"
)

df_weather_clean["rainfall_mm"] = (
    rain * df_weather_clean["rain_unit_clean"].map({
        "mm": 1,
        "inches": 25.4
    })
).where(df_weather_clean["rainfall_status"].eq("Valid"))

display(
    df_weather_clean["rainfall_status"]
    .value_counts()
    .to_frame("row_count")
)

display(df_weather_clean["rainfall_mm"].describe())

,row_count
rainfall_status,
Valid,12691
Negative rainfall,1518
Missing rainfall,791


count    12691.000000
mean        25.029117
std         14.459714
min          0.000000
25%         12.600000
50%         24.900000
75%         37.592000
max         50.038000
Name: rainfall_mm, dtype: float64

## Validating humidity

In [13]:
raw_humidity = (
    df_weather_clean["humidity_percent"]
    .astype("string").str.strip().replace("", pd.NA)
)

humidity = pd.to_numeric(raw_humidity, errors="coerce")

df_weather_clean["humidity_status"] = np.select(
    [
        raw_humidity.isna().to_numpy(dtype=bool),
        humidity.isna().to_numpy(dtype=bool),
        (~humidity.between(0, 100)).fillna(False).to_numpy(dtype=bool)
    ],
    [
        "Missing humidity",
        "Unparseable humidity",
        "Outside 0–100%"
    ],
    default="Valid"
)

df_weather_clean["humidity_percent_clean"] = humidity.where(
    df_weather_clean["humidity_status"].eq("Valid")
)

display(
    df_weather_clean["humidity_status"]
    .value_counts()
    .to_frame("row_count")
)

,row_count
humidity_status,
Valid,13500
Missing humidity,1500


## Inspecting weather timestamps and format labels

In [14]:
timestamps = (
    df_weather_clean["timestamp"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

print("Timestamp formats:")
display(
    timestamps.str.replace(r"\d", "D", regex=True)
    .value_counts(dropna=False)
    .to_frame("row_count")
)

print("\nSample timestamps:")
display(
    timestamps.dropna()
    .drop_duplicates()
    .head(20)
    .to_frame("timestamp")
)

Timestamp formats:


,row_count
timestamp,
DDDD-DD-DD DD:DD:DD IST,6038
DDDD-DD-DD DD:DD:DD UTC,5141
<NA>,1555
DDDD-DD-DD DD:DD:DD,601
DD/DD/DDDD DD:DD,430
DDDD-DD-DDTDD:DD:DD,300
DD/DD/DDDD,226
DD-DD-DDDD DD:DD PM,182
DD-DD-DDDD DD:DD AM,165



Sample timestamps:


,timestamp
0,28/08/2026
1,2026-07-03 07:08:18 IST
3,2026-07-09 11:08:41 UTC
4,2026-06-09 13:06:26 IST
5,17/03/2026
6,07-01-2026 05:48 PM
7,2026-06-12 09:52:51 IST
8,2026-06-02 16:00:04 UTC
10,2026-08-23 07:42:11 IST
11,2026-09-08 14:45:54 UTC


## Cleaning timestamps

In [15]:
timestamps = (
    df_weather_clean["timestamp"]
    .astype("string").str.strip().replace("", pd.NA)
)

utc_mask = timestamps.str.endswith(" UTC", na=False)
ist_mask = timestamps.str.endswith(" IST", na=False)

# A timezone-aware column for standardized timestamps.
df_weather_clean["timestamp_ist"] = pd.Series(
    pd.NaT,
    index=df_weather_clean.index,
    dtype="datetime64[ns, Asia/Kolkata]"
)

utc_values = pd.to_datetime(
    timestamps.loc[utc_mask].str.replace(r" UTC$", "", regex=True),
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce",
    utc=True
)

ist_values = pd.to_datetime(
    timestamps.loc[ist_mask].str.replace(r" IST$", "", regex=True),
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
).dt.tz_localize("Asia/Kolkata")

df_weather_clean.loc[utc_mask, "timestamp_ist"] = (
    utc_values.dt.tz_convert("Asia/Kolkata")
)

df_weather_clean.loc[ist_mask, "timestamp_ist"] = ist_values

print(
    "Explicit-zone timestamps parsed:",
    df_weather_clean["timestamp_ist"].notna().sum()
)
print(
    "Failed explicit-zone timestamps:",
    ((utc_mask | ist_mask) & df_weather_clean["timestamp_ist"].isna()).sum()
)

display(
    df_weather_clean.loc[
        utc_mask, ["timestamp", "timestamp_ist"]
    ].head(5)
)

Explicit-zone timestamps parsed: 11179
Failed explicit-zone timestamps: 0


,timestamp,timestamp_ist
3,2026-07-09 11:08:41 UTC,2026-07-09 16:38:41+05:30
8,2026-06-02 16:00:04 UTC,2026-06-02 21:30:04+05:30
11,2026-09-08 14:45:54 UTC,2026-09-08 20:15:54+05:30
12,2026-03-25 23:52:33 UTC,2026-03-26 05:22:33+05:30
15,2026-03-21 12:06:58 UTC,2026-03-21 17:36:58+05:30


## Inspecting numeric dates

In [16]:
for separator in ["/", "-"]:
    parts = timestamps.str.extract(
        rf"^(\d{{2}})[{separator}](\d{{2}})[{separator}]\d{{4}}(?:\s|$)"
    )

    first = pd.to_numeric(parts[0], errors="coerce")
    second = pd.to_numeric(parts[1], errors="coerce")

    print(
        f"{separator}: "
        f"first number > 12 = {first.gt(12).sum()}, "
        f"second number > 12 = {second.gt(12).sum()}"
    )

/: first number > 12 = 385, second number > 12 = 0
-: first number > 12 = 0, second number > 12 = 192


## Parsing unlabelled values 

In [17]:
unlabelled = timestamps.notna() & ~utc_mask & ~ist_mask

formats = {
    r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}": "%Y-%m-%d %H:%M:%S",
    r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}": "%Y-%m-%dT%H:%M:%S",
    r"\d{2}/\d{2}/\d{4} \d{2}:\d{2}": "%d/%m/%Y %H:%M",
    r"\d{2}/\d{2}/\d{4}": "%d/%m/%Y",
    r"\d{2}-\d{2}-\d{4} \d{2}:\d{2} [AP]M": "%m-%d-%Y %I:%M %p",
    r"\d{2}-[A-Za-z]{3}-\d{4} \d{2}:\d{2}:\d{2}": "%d-%b-%Y %H:%M:%S"
}

parsed = pd.Series(pd.NaT, index=df_weather_clean.index)

for pattern, date_format in formats.items():
    mask = unlabelled & timestamps.str.fullmatch(pattern, na=False)
    parsed.loc[mask] = pd.to_datetime(
        timestamps.loc[mask], format=date_format, errors="coerce"
    )

date_only = timestamps.str.fullmatch(r"\d{2}/\d{2}/\d{4}", na=False)

# Clear only unlabelled rows so this step can be rerun safely.
df_weather_clean.loc[unlabelled, "timestamp_ist"] = pd.NaT

has_time = unlabelled & ~date_only & parsed.notna()
df_weather_clean.loc[has_time, "timestamp_ist"] = (
    parsed.loc[has_time].dt.tz_localize("Asia/Kolkata")
)

# Daily reporting date: use converted IST date where available.
df_weather_clean["weather_date"] = (
    df_weather_clean["timestamp_ist"]
    .dt.tz_localize(None).dt.normalize()
)

df_weather_clean.loc[date_only, "weather_date"] = (
    parsed.loc[date_only].dt.normalize()
)

df_weather_clean["timestamp_status"] = np.select(
    [
        timestamps.isna().to_numpy(dtype=bool),
        df_weather_clean["weather_date"].isna().to_numpy(dtype=bool),
        date_only.to_numpy(dtype=bool),
        utc_mask.to_numpy(dtype=bool),
        ist_mask.to_numpy(dtype=bool)
    ],
    [
        "Missing",
        "Invalid timestamp",
        "Date only: assumed local date",
        "Explicit UTC converted to IST",
        "Explicit IST"
    ],
    default="Unlabelled time: assumed IST"
)

# Flag ambiguous day/month interpretations separately.
parts = timestamps.str.extract(r"^(\d{2})[/-](\d{2})[/-]\d{4}(?:\s|$)")
first = pd.to_numeric(parts[0], errors="coerce")
second = pd.to_numeric(parts[1], errors="coerce")

df_weather_clean["date_format_assumed"] = (
    first.between(1, 12)
    & second.between(1, 12)
    & first.ne(second)
)

display(df_weather_clean["timestamp_status"].value_counts())

timestamp_status
Explicit IST                     6038
Explicit UTC converted to IST    5141
Unlabelled time: assumed IST     2040
Missing                          1555
Date only: assumed local date     226
Name: count, dtype: int64

## Inspecting sensor ids

In [18]:
df_weather_clean["sensor_id_clean"] = (
    df_weather_clean["sensor_id"]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace("", pd.NA)
)

sensors = df_weather_clean["sensor_id_clean"]

print("Missing sensor IDs:", sensors.isna().sum())
print("Unique sensors:", sensors.nunique())

display(
    sensors.value_counts(dropna=False)
    .to_frame("reading_count")
)

print("\nSensor ID values:")
print(sorted(sensors.dropna().unique()))

Missing sensor IDs: 0
Unique sensors: 51


,reading_count
sensor_id_clean,
UNKNOWN,715
SEN050,338
SEN005,335
SEN025,312
SEN029,308
SEN017,308
SEN010,307
SEN008,305
SEN007,304



Sensor ID values:
['SEN001', 'SEN002', 'SEN003', 'SEN004', 'SEN005', 'SEN006', 'SEN007', 'SEN008', 'SEN009', 'SEN010', 'SEN011', 'SEN012', 'SEN013', 'SEN014', 'SEN015', 'SEN016', 'SEN017', 'SEN018', 'SEN019', 'SEN020', 'SEN021', 'SEN022', 'SEN023', 'SEN024', 'SEN025', 'SEN026', 'SEN027', 'SEN028', 'SEN029', 'SEN030', 'SEN031', 'SEN032', 'SEN033', 'SEN034', 'SEN035', 'SEN036', 'SEN037', 'SEN038', 'SEN039', 'SEN040', 'SEN041', 'SEN042', 'SEN043', 'SEN044', 'SEN045', 'SEN046', 'SEN047', 'SEN048', 'SEN049', 'SEN050', 'UNKNOWN']


## Flagging missing ids

In [20]:
df_weather_clean["sensor_id_clean"] = (
    df_weather_clean["sensor_id_clean"]
    .replace("UNKNOWN", pd.NA)
)

df_weather_clean["sensor_id_status"] = np.where(
    df_weather_clean["sensor_id_clean"].isna(),
    "Missing or unknown sensor",
    "Present"
)

print("Known sensors:", df_weather_clean["sensor_id_clean"].nunique())
display(
    df_weather_clean["sensor_id_status"]
    .value_counts()
    .to_frame("reading_count")
)

Known sensors: 50


,reading_count
sensor_id_status,
Present,14285
Missing or unknown sensor,715


## Checking sensor timestamps pair

In [19]:
complete = (
    df_weather_clean["sensor_id_clean"].notna()
    & df_weather_clean["timestamp_ist"].notna()
)

repeated = (
    df_weather_clean.loc[complete]
    .duplicated(
        subset=["sensor_id_clean", "timestamp_ist"],
        keep=False
    )
)

print("Rows with repeated sensor–timestamp pairs:", repeated.sum())

Rows with repeated sensor–timestamp pairs: 2


## Checking if they contain identical reading or conflicting measurements

In [21]:
complete = (
    df_weather_clean["sensor_id_clean"].notna()
    & df_weather_clean["timestamp_ist"].notna()
)

known_readings = df_weather_clean.loc[complete]

repeated_rows = known_readings.loc[
    known_readings.duplicated(
        ["sensor_id_clean", "timestamp_ist"],
        keep=False
    )
]

display(
    repeated_rows[
        [
            "sensor_id",
            "timestamp",
            "sensor_id_clean",
            "timestamp_ist",
            "timestamp_status",
            "temperature_c",
            "rainfall_mm",
            "humidity_percent_clean"
        ]
    ].sort_values(["sensor_id_clean", "timestamp_ist"])
)

,sensor_id,timestamp,sensor_id_clean,timestamp_ist,timestamp_status,temperature_c,rainfall_mm,humidity_percent_clean
9494,SEN050,2026-09-09 06:53:34 IST,SEN050,2026-09-09 06:53:34+05:30,Explicit IST,29.1,15.7,90.0
10740,SEN050,2026-09-09 06:53:34 IST,SEN050,2026-09-09 06:53:34+05:30,Explicit IST,15.222222,31.4,41.0


## Flagging these readings

In [22]:
complete = (
    df_weather_clean["sensor_id_clean"].notna()
    & df_weather_clean["timestamp_ist"].notna()
)

df_weather_clean["sensor_timestamp_conflict"] = False

df_weather_clean.loc[complete, "sensor_timestamp_conflict"] = (
    df_weather_clean.loc[complete]
    .duplicated(
        ["sensor_id_clean", "timestamp_ist"],
        keep=False
    )
)

print(
    "Conflicting readings flagged:",
    df_weather_clean["sensor_timestamp_conflict"].sum()
)

Conflicting readings flagged: 2


## Final validation

In [23]:
assert len(df_weather_clean) == 15000, "Unexpected row count."

assert df_weather_clean["temperature_c"].notna().all()
assert df_weather_clean["temperature_unit_clean"].isin(["C", "F"]).all()

valid_rain = df_weather_clean["rainfall_status"].eq("Valid")
rain = df_weather_clean["rainfall_mm"]

assert rain.loc[valid_rain].notna().all()
assert rain.loc[valid_rain].ge(0).all()
assert rain.loc[~valid_rain].isna().all()

valid_humidity = df_weather_clean["humidity_status"].eq("Valid")
humidity = df_weather_clean["humidity_percent_clean"]

assert humidity.loc[valid_humidity].notna().all()
assert humidity.loc[valid_humidity].between(0, 100).all()
assert humidity.loc[~valid_humidity].isna().all()

date_only = df_weather_clean["timestamp_status"].eq(
    "Date only: assumed local date"
)

assert df_weather_clean.loc[date_only, "timestamp_ist"].isna().all()
assert df_weather_clean.loc[date_only, "weather_date"].notna().all()

assert not df_weather_clean["timestamp_status"].eq(
    "Invalid timestamp"
).any()

assert df_weather_clean["sensor_timestamp_conflict"].sum() == 2

print("Weather validation passed.")
print("Total readings:", len(df_weather_clean))
print("Known sensors:", df_weather_clean["sensor_id_clean"].nunique())
print("Valid rainfall readings:", int(valid_rain.sum()))
print("Valid humidity readings:", int(valid_humidity.sum()))
print("Full timestamps:", df_weather_clean["timestamp_ist"].notna().sum())
print("Available reporting dates:", df_weather_clean["weather_date"].notna().sum())

Weather validation passed.
Total readings: 15000
Known sensors: 50
Valid rainfall readings: 12691
Valid humidity readings: 13500
Full timestamps: 13219
Available reporting dates: 13445


## Exporting file

In [24]:
from pathlib import Path

export_columns = {
    "sensor_id": "sensor_id_clean",
    "timestamp_ist": "timestamp_ist",
    "weather_date": "weather_date",
    "temperature_c": "temperature_c",
    "rainfall_mm": "rainfall_mm",
    "humidity_percent": "humidity_percent_clean",
    "sensor_id_status": "sensor_id_status",
    "timestamp_status": "timestamp_status",
    "date_format_assumed": "date_format_assumed",
    "rainfall_status": "rainfall_status",
    "humidity_status": "humidity_status",
    "sensor_timestamp_conflict": "sensor_timestamp_conflict",
    "source_sensor_id": "sensor_id",
    "source_timestamp": "timestamp",
    "source_temperature": "temperature",
    "source_temp_unit": "temp_unit",
    "source_rainfall": "rainfall",
    "source_rain_unit": "rain_unit",
    "source_humidity_percent": "humidity_percent"
}

weather_export = df_weather_clean[
    list(export_columns.values())
].copy()

weather_export.columns = list(export_columns.keys())

# Preserve the timezone offset in exported timestamps.
weather_export["timestamp_ist"] = weather_export["timestamp_ist"].map(
    lambda value: value.isoformat() if pd.notna(value) else pd.NA
)

weather_export["weather_date"] = (
    weather_export["weather_date"].dt.strftime("%Y-%m-%d")
)

output_folder = Path(
    "/Users/rohankhanna/Documents/datathon/cleaned_data"
)
output_folder.mkdir(parents=True, exist_ok=True)

output_file = output_folder / "weather_sensors_cleaned.csv"

weather_export.to_csv(
    output_file,
    index=False,
    na_rep="NA",
    encoding="utf-8-sig"
)

print("Saved:", output_file)

Saved: /Users/rohankhanna/Documents/datathon/cleaned_data/weather_sensors_cleaned.csv
